In [2]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications.efficientnet import preprocess_input
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
train_real = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_train/face_real/*.jpg",shuffle=False)
train_fake = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_train/face_fake/*.jpg",shuffle=False)
validation_real = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_validation/face_real/*.jpg",shuffle=False)
validation_fake = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_validation/face_fake/*.jpg",shuffle=False)
test_real = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_test/face_real/*.jpg",shuffle=False)
test_fake = tf.data.Dataset.list_files("/content/drive/MyDrive/face_detection_test/face_fake/*.jpg",shuffle=False)


In [4]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    return img

train_real = train_real.map(load_image)
train_fake = train_fake.map(load_image)

validation_real = validation_real.map(load_image)
validation_fake = validation_fake.map(load_image)

test_real = test_real.map(load_image)
test_fake = test_fake.map(load_image)

In [5]:
def add_label(image, label):
    return image, label

train_real = train_real.map(lambda x: add_label(x, 0))
train_fake = train_fake.map(lambda x: add_label(x, 1))

validation_real = validation_real.map(lambda x: add_label(x, 0))
validation_fake = validation_fake.map(lambda x: add_label(x, 1))

test_real = test_real.map(lambda x: add_label(x, 0))
test_fake = test_fake.map(lambda x: add_label(x, 1))

In [6]:
train_dataset = train_real.concatenate(train_fake)
validation_dataset = validation_real.concatenate(validation_fake)
test_dataset = test_real.concatenate(test_fake)

In [7]:
train_dataset = train_dataset.shuffle(6400)

In [8]:
def preprocess(image, label):
    image = preprocess_input(image)
    return image, label

train_dataset = train_dataset.map(preprocess)

validation_dataset = validation_dataset.map(preprocess)

test_dataset = test_dataset.map(preprocess)

In [9]:
BATCH_SIZE = 32

train_dataset = train_dataset.batch(BATCH_SIZE)
validation_dataset = validation_dataset.batch(BATCH_SIZE)
test_dataset = test_dataset.batch(BATCH_SIZE)
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

In [10]:
model = tf.keras.models.load_model("/content/drive/MyDrive/models/efficientnet_stage2.keras")

base_model = model.layers[1]
base_model.trainable = True

for layer in base_model.layers[:-15]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/efficientnet_finetuned4.keras",
    monitor="val_accuracy",
    save_best_only=True
)

history_finetune = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 2557s 3s/step - accuracy: 0.6097 - loss: 0.6563 - val_accuracy: 0.6250 - val_loss: 0.6466
Epoch 2/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 28s 60ms/step - accuracy: 0.6869 - loss: 0.5913 - val_accuracy: 0.6481 - val_loss: 0.6262
Epoch 3/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 29s 60ms/step - accuracy: 0.7222 - loss: 0.5502 - val_accuracy: 0.6569 - val_loss: 0.6197
Epoch 4/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 28s 60ms/step - accuracy: 0.7486 - loss: 0.5147 - val_accuracy: 0.6781 - val_loss: 0.5989
Epoch 5/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 28s 57ms/step - accuracy: 0.7713 - loss: 0.4860 - val_accuracy: 0.6762 - val_loss: 0.5961
Epoch 6/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 28s 56ms/step - accuracy: 0.7811 - loss: 0.4630 - val_accuracy: 0.6725 - val_loss: 0.6162
Epoch 7/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 41s 62ms/step - accuracy: 0.8061 - loss: 0.4343 - val_accuracy: 0.6831 - val_loss: 0.5942
Epoch 8/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 29s 62ms/step - accuracy: 0.8163 - loss: 0.4083 - 

In [11]:
model.evaluate(test_dataset)

63/63 ━━━━━━━━━━━━━━━━━━━━ 635s 10s/step - accuracy: 0.6880 - loss: 0.5848


[0.5847940444946289, 0.6880000233650208]